# Phase A: local hardware emulation (Deliverable 3)

`AerSimulator.from_backend` turns a fake backend (a calibration snapshot of a real IBM chip: coupling map, native gates, gate and readout errors, T1/T2) into a noise model and runs the transpiled circuit locally.

* `FakeManilaV2` has 5 qubits; the generalized circuit needs 2n+3 (11 for N = 15). `FakeTorino` (133 qubits, Heron r1) matches the family of today's cloud QPUs.
* The semiclassical circuit uses mid-circuit measurement, reset and classically controlled phases. Aer runs these natively; `ensure_control_flow` declares them on the fake backend if its snapshot omits them.
* The generalized circuit is thousands of gates long, so transpilation and noisy simulation take minutes, not seconds. Start with N = 15.

In [2]:
RUN_CORE_DEMO = True                # load the core functions only, skip its demo/self-test cells
%run shor_core_generalized.ipynb

N, A = 15, 7                           # any odd composite N (not a prime power) and 1 < A < N, gcd(A, N) = 1
T = 3                               # counting precision t; None = 2n
MODE = "semiclassical"                 # "semiclassical" = Beauregard 2n+3 qubits; "full" = t-qubit counting register

qc = build_shor_circuit(N, A, T, MODE, SKIP_IDENTITY)
t = qc.num_clbits
tag = run_tag(N, A, t, MODE)
print(f"{qc.name}: {qc.num_qubits} qubits, t = {t}")

pre-checks: (None, 'ok')
n = 4, t = 8, M = 2^t = 256 >= N^2 = 225: True
a^(2^j) mod N, j = 0..7: [7, 4, 1, 1, 1, 1, 1, 1]
qubits: 11  (2n+3 = 11)
logical ops: {'cp': 1394, 'h': 376, 'cx': 128, 'p': 80, 'x': 33, 'if_else': 28, 'measure': 8, 'cswap': 8, 'reset': 7}
two-qubit gates (cswap counted separately): 1522
add: OK
controlled-modular-add: OK
controlled-U_a (a = 7, N = 15) for all x < N: OK
Hellinger fidelity, noiseless circuit vs exact formula: 1.0000
success rate: sampled 0.506, exact 0.500
     y  shots      y/M  continued fraction -> convergents -> outcome
    64   1040  0.25000  [0, 4] -> ['0/1', '1/4'] -> FACTORS (3, 5): r = 4, a^(r/2) = 4, gcd(3, N) = 3, gcd(5, N) = 5
   192   1031  0.75000  [0, 1, 3] -> ['0/1', '1/1', '3/4'] -> FACTORS (3, 5): r = 4, a^(r/2) = 4, gcd(3, N) = 3, gcd(5, N) = 5
   128   1023  0.50000  [0, 2] -> ['0/1', '1/2'] -> fail: no convergent denominator d < N satisfies a^d = 1 mod N
     0   1002  0.00000  [0] -> ['0/1'] -> fail: y = 0 carries no period 

In [3]:
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import fake_provider

FAKE_BACKEND = "FakeTorino"
SHOTS = 2048
TRANSPILE_SEEDS = range(3)

## 1. Transpile (optimisation level 3, best of several seeds)

In [4]:
backend = getattr(fake_provider, FAKE_BACKEND)()
assert backend.num_qubits >= qc.num_qubits, f"{FAKE_BACKEND} has only {backend.num_qubits} qubits"
if MODE == "semiclassical":
    ensure_control_flow(backend)

isa, seed = best_isa_circuit(qc, backend, TRANSPILE_SEEDS)
logical, physical = circuit_report(qc), circuit_report(isa)
print(f"Logical : depth {logical['depth']}, two-qubit gates {logical['two_qubit_gates']}, ops {logical['ops']}")
print(f"ISA     : depth {physical['depth']}, two-qubit gates {physical['two_qubit_gates']}  (seed {seed})")

Logical : depth 1252, two-qubit gates 1522, ops {'cp': 1394, 'h': 366, 'cx': 128, 'p': 80, 'x': 33, 'cswap': 8, 'measure': 3, 'if_else': 3, 'reset': 2}
ISA     : depth 13643, two-qubit gates 6188  (seed 0)


## 2. Noisy simulation with the device noise model

In [5]:
sim = AerSimulator.from_backend(backend)
counts = sim.run(isa, shots=SHOTS, seed_simulator=11).result().get_counts()

## 3. Quick metrics and save (Phase C makes the full comparison)

In [6]:
ideal = exact_distribution(A, N, t)
print(f"Hellinger fidelity vs ideal: {hellinger_fidelity(ideal, counts):.4f}")
print(f"Factoring success rate     : {success_rate(counts, t, A, N):.4f}   (ideal {success_rate(ideal, t, A, N):.3f})")

save_json(f"local_counts_{tag}.json", {
    "backend": FAKE_BACKEND, "N": N, "a": A, "t": t, "mode": MODE, "shots": SHOTS,
    "seed_transpiler": seed, "isa_stats": physical, "counts": counts})
print(f"Saved {results_path(f'local_counts_{tag}.json')}")

Hellinger fidelity vs ideal: 0.9940
Factoring success rate     : 0.5088   (ideal 0.500)
Saved results/local_counts_N15_a7_t3_semiclassical.json
